# Add episodes on a free cloud GPU

Adding episodes is the only step of anime-voice-collector that needs an NVIDIA graphics card. This notebook does it on Google Colab's free GPU; labeling afterwards runs on any computer.

1. **Use a GPU**: Runtime → Change runtime type → T4 GPU.
2. **Put the files in Google Drive**: make the folder `avoc-workspace/input` in My Drive and upload the videos and their main subtitles there. The episode number in each file name pairs them, as in the app's "Several, one after another". Subtitles only shown while labeling can go in `avoc-workspace/input/display`.
3. **For automatic timing**, add your Hugging Face token under Secrets (the key icon on the left) with the name `HF_TOKEN`, and let this notebook use it ([why it is needed](https://github.com/raindrops0/anime-voice-collector/blob/main/docs/guide/install.md#hugging-face-token-for-automatic-timing)). Without one, give the offset in `OPTIONS` below, such as `--offset 0`.
4. Runtime → Run all.

Each finished episode is copied back to Google Drive at once. If the session ends, run everything again: finished episodes are skipped and an unfinished one carries on.

In [ ]:
FOLDER = "avoc-workspace"  # in My Drive
OPTIONS = ""  # given to every `avoc add`, such as "--spoken ja" or "--offset 0"
REPO = "https://github.com/raindrops0/anime-voice-collector"
BRANCH = "main"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!test -d /content/avoc || git clone --depth 1 --branch $BRANCH $REPO /content/avoc
%cd /content/avoc
!uv sync --locked

In [ ]:
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret: automatic timing will not work unless OPTIONS gives --offset")
!uv run python scripts/colab/add_all.py "/content/drive/MyDrive/$FOLDER" /content/work $OPTIONS

## Label on your own computer

Download the `avoc-workspace` folder from Google Drive (right-click → Download; Drive splits a big folder into several zip files, so unzip them all into one folder), or let Google Drive for desktop sync it. Then [install anime-voice-collector](https://github.com/raindrops0/anime-voice-collector/blob/main/docs/guide/install.md) and open the folder:

```
uv run avoc app avoc-workspace
```

The videos stay in `input/`, inside the workspace, so the app finds them after the move. The first time it opens each episode, it makes the copy the page plays again, which takes about a minute per episode.